In [1]:
!pip -q install datasets pyarrow xxhash pyspark

In [2]:
import os
import re
import gc
import time
import math
import hashlib
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq
from datasets import load_dataset
from tqdm.auto import tqdm
import xxhash

In [3]:
# ====== CONFIG ======
HF_DATASET_NAME = "common-pile/peS2o_filtered"
HF_SPLIT = "train"

SOURCE_SUBSTR = "s2orc"

TARGET_DOCS = 1000000

MIN_WORDS = 500



# 5-gram là lựa chọn hợp lý để scale benchmark trước
NGRAM_N = 5

# Đây là mẹo để fit Kaggle, không phải paper bắt buộc:
# cap số hashed shingles unique mỗi doc để đỡ nổ disk/RAM
MAX_UNIQUE_SHINGLES = 1024

TEXT_OUT_DIR = Path("/kaggle/working/pes2o_scale_text_parquet")
HASH_OUT_DIR = Path("/kaggle/working/pes2o_scale_hashed5_parquet")
SUBSET_OUT_DIR = Path("/kaggle/working/pes2o_scale_subsets_hashed5")

LOG_EVERY = 10_000

In [4]:
TEXT_SCHEMA = pa.schema([
    ("doc_id", pa.int64()),
    ("source", pa.string()),
    ("text_light_clean", pa.string()),
    ("n_words", pa.int32()),
    ("n_chars", pa.int32()),
    ("exact_hash", pa.string()),
])

def ensure_empty_dir(path: Path):
    path.mkdir(parents=True, exist_ok=True)

def write_text_chunk(rows, part_idx: int, out_dir: Path):
    table = pa.Table.from_pylist(rows, schema=TEXT_SCHEMA)
    out_path = out_dir / f"part-{part_idx:05d}.parquet"
    pq.write_table(table, out_path, compression="zstd")
    return out_path

In [5]:
def count_words(text: str) -> int:
    return len(text.split())

def exact_md5(text: str) -> str:
    return hashlib.md5(text.encode("utf-8", errors="ignore")).hexdigest()

In [6]:
READ_BATCH_SIZE = 4096
ROWS_PER_FILE = 10000

In [7]:
def batched_iter(iterable, batch_size):
    batch = []
    for x in iterable:
        batch.append(x)
        if len(batch) >= batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

In [8]:
def process_examples_batch(examples, start_doc_id):
    rows = []
    next_doc_id = start_doc_id

    for ex in examples:
        src = str(ex.get("source", "")).lower()
        if SOURCE_SUBSTR not in src:
            continue

        text = ex.get("text", None)
        if not text or not isinstance(text, str):
            continue

        text_light = text  # hoặc light_clean_text(text) nếu bạn muốn
        if not text_light:
            continue

        n_words = count_words(text_light)
        if n_words < MIN_WORDS:
            continue

        rows.append({
            "doc_id": next_doc_id,
            "source": src,
            "text_light_clean": text_light,
            "n_words": n_words,
            "n_chars": len(text_light),
            "exact_hash": exact_md5(text_light),
        })
        next_doc_id += 1

    return rows, next_doc_id

In [9]:
ensure_empty_dir(TEXT_OUT_DIR)

dataset = load_dataset(
    HF_DATASET_NAME,
    split=HF_SPLIT,
    streaming=True,
)

buffer = []
part_idx = 0
kept = 0
seen = 0

t0 = time.perf_counter()

for ex_batch in batched_iter(dataset, READ_BATCH_SIZE):
    seen += len(ex_batch)

    rows, next_doc_id = process_examples_batch(ex_batch, kept)
    buffer.extend(rows)
    kept = next_doc_id

    while len(buffer) >= ROWS_PER_FILE:
        write_text_chunk(buffer[:ROWS_PER_FILE], part_idx, TEXT_OUT_DIR)
        buffer = buffer[ROWS_PER_FILE:]
        part_idx += 1
        gc.collect()

    if kept and kept % LOG_EVERY < READ_BATCH_SIZE:
        elapsed = time.perf_counter() - t0
        print(f"[kept={kept:,}] [seen={seen:,}] [elapsed={elapsed/60:.1f} min]")

    if kept >= TARGET_DOCS:
        break

if buffer:
    write_text_chunk(buffer, part_idx, TEXT_OUT_DIR)

elapsed = time.perf_counter() - t0
print(f"Done. kept={kept:,}, seen={seen:,}, elapsed={elapsed/60:.2f} min")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/91 [00:00<?, ?it/s]

[kept=4,091] [seen=4,096] [elapsed=0.1 min]
[kept=12,277] [seen=12,288] [elapsed=0.4 min]
[kept=20,465] [seen=20,480] [elapsed=0.6 min]
[kept=32,743] [seen=32,768] [elapsed=1.0 min]
[kept=40,927] [seen=40,960] [elapsed=1.2 min]
[kept=53,202] [seen=53,248] [elapsed=1.6 min]
[kept=61,388] [seen=61,440] [elapsed=1.8 min]
[kept=73,663] [seen=73,728] [elapsed=2.2 min]
[kept=81,849] [seen=81,920] [elapsed=2.4 min]
[kept=90,034] [seen=90,112] [elapsed=2.7 min]
[kept=102,314] [seen=102,400] [elapsed=3.0 min]
[kept=110,500] [seen=110,592] [elapsed=3.2 min]
[kept=122,779] [seen=122,880] [elapsed=3.7 min]
[kept=130,964] [seen=131,072] [elapsed=3.9 min]
[kept=143,243] [seen=143,360] [elapsed=4.2 min]
[kept=151,433] [seen=151,552] [elapsed=4.5 min]
[kept=163,721] [seen=163,840] [elapsed=4.8 min]
[kept=171,907] [seen=172,032] [elapsed=5.1 min]
[kept=180,091] [seen=180,224] [elapsed=5.3 min]
[kept=192,363] [seen=192,512] [elapsed=5.6 min]
[kept=200,548] [seen=200,704] [elapsed=5.9 min]
[kept=212,829]

In [10]:
# %%
from pathlib import Path
import os

parquet_files = sorted(TEXT_OUT_DIR.glob("part-*.parquet"))
print("num parquet files:", len(parquet_files))

total_bytes = sum(p.stat().st_size for p in parquet_files)
print("total size (GB):", round(total_bytes / (1024**3), 3))

if parquet_files:
    print("first file:", parquet_files[0].name)
    print("last file :", parquet_files[-1].name)

num parquet files: 101
total size (GB): 9.39
first file: part-00000.parquet
last file : part-00100.parquet


In [11]:
# %%
import pyarrow.parquet as pq
from tqdm.auto import tqdm

total_rows = 0
for fp in tqdm(parquet_files):
    pf = pq.ParquetFile(fp)
    total_rows += pf.metadata.num_rows

print("total rows in parquet:", total_rows)
print("kept from pipeline    :", kept)

  0%|          | 0/101 [00:00<?, ?it/s]

total rows in parquet: 1002782
kept from pipeline    : 1002782


In [12]:
# %%
if parquet_files:
    sample_table = pq.read_table(
        parquet_files[0],
        columns=["doc_id", "source", "n_words", "n_chars", "exact_hash"]
    )
    print(sample_table.schema)
    display(sample_table.to_pandas().head(10))

doc_id: int64
source: string
n_words: int32
n_chars: int32
exact_hash: string


,doc_id,source,n_words,n_chars,exact_hash
0,0,pes2o/s2orc,4272,26066,cc282a0b418ba334f5f819eda76726a1
1,1,pes2o/s2orc,4011,26627,a2f9b56e5f1a40b3d9c54aa17f47fd49
2,2,pes2o/s2orc,1969,12245,2b8812edfc308aff07f5e4cc0f3b946c
3,3,pes2o/s2orc,3475,23780,daa4a317e1690e5d2ad36f367cbd680c
4,4,pes2o/s2orc,3935,25475,519318327afb8fbe974c85c45d7b4638
5,5,pes2o/s2orc,6501,40142,977df74a2bac4b8597bb83393a84c162
6,6,pes2o/s2orc,9838,61185,3831ec452f17a89ba640f3bafc0bfea8
7,7,pes2o/s2orc,3733,24318,e5f0689e0a7cf0fc0acb9d831b2cf54f
8,8,pes2o/s2orc,6247,40188,31ef982a3a101a98cdde525ebde6d3ea
9,9,pes2o/s2orc,5672,34187,129f15d19b29c8c92c60d05301b35f50


In [13]:
# %%
if parquet_files:
    sample_text_table = pq.read_table(
        parquet_files[0],
        columns=["doc_id", "text_light_clean"]
    )
    sample_df = sample_text_table.to_pandas().head(3)

    for i, row in sample_df.iterrows():
        print("=" * 100)
        print("doc_id:", row["doc_id"])
        print(row["text_light_clean"][:1500])
        print()

doc_id: 0
Subleading contributions to the nuclear scalar isoscalar currents

We extend our recent analyses of the nuclear vector, axial-vector and pseudoscalar currents and derive the leading one-loop corrections to the two-nucleon scalar current operator in the framework of chiral effective field theory using the method of unitary transformation. We also show that the scalar current operators at zero momentum transfer are directly related to the quark mass dependence of the nuclear forces.


I. INTRODUCTION
The first principles description of nuclei, nuclear matter and reactions is one of the great challenges in contemporary physics with applications ranging from low-energy searches for physics beyond the Standard Model (SM) to properties of neutron stars and neutron star mergers. The currently most efficient and feasible approach along this line relies on the application of suitably taylored effective field theories (EFTs). In particular, an extension of chiral perturbation theory to

In [14]:
# %%
import pandas as pd

word_stats = []

for fp in tqdm(parquet_files[:min(5, len(parquet_files))]):  # check vài file đầu thôi
    df = pq.read_table(fp, columns=["n_words", "n_chars"]).to_pandas()
    word_stats.append({
        "rows": len(df),
        "min_words": int(df["n_words"].min()),
        "mean_words": float(df["n_words"].mean()),
        "max_words": int(df["n_words"].max()),
        "mean_chars": float(df["n_chars"].mean()),
    })

display(pd.DataFrame(word_stats))

  0%|          | 0/5 [00:00<?, ?it/s]

,rows,min_words,mean_words,max_words,mean_chars
0,10000,513,4593.4057,33715,29826.7219
1,10000,539,4574.1987,38942,29721.0064
2,10000,547,4569.9858,30630,29660.3608
3,10000,503,4647.1736,34475,30203.2672
4,10000,502,4631.8791,39769,30091.8888


In [15]:
# %%
!pip -q install sentence-transformers pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.7 MB/s eta 0:00:00


In [16]:
# %%
import os
import gc
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import psutil
import pynvml
import torch

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

In [17]:
# %%
RUN_NAME = "scale_minilm_gpu_50k"
RUN_DOC_LIMIT = 50_000

RUN_DIR = Path(f"/kaggle/working/{RUN_NAME}")
EMB_DIR = RUN_DIR / "embeddings"
NN_DIR = RUN_DIR / "neighbors"

for d in [RUN_DIR, EMB_DIR, NN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

PARQUET_READ_BATCH = 2000
ENCODE_BATCH_SIZE = 256

TOPK = 10
QUERY_BLOCK_SIZE = 1024
REF_BLOCK_SIZE = 8192

NORMALIZE_EMBEDDINGS = True

In [18]:
# %%
def make_embed_view(text: str, head_words=64, mid_words=64, tail_words=64) -> str:
    toks = text.split()
    if not toks:
        return ""

    total_need = head_words + mid_words + tail_words
    if len(toks) <= total_need:
        return " ".join(toks)

    head = toks[:head_words]

    mid_start = max(0, len(toks) // 2 - mid_words // 2)
    mid = toks[mid_start: mid_start + mid_words]

    tail = toks[-tail_words:]

    return " ".join(head + ["[MIDDLE]"] + mid + ["[TAIL]"] + tail)

In [19]:
# %%
process = psutil.Process(os.getpid())

pynvml.nvmlInit()
gpu_handle = pynvml.nvmlDeviceGetHandleByIndex(0)

def ram_used_gb():
    return process.memory_info().rss / (1024**3)

def gpu_used_gb():
    info = pynvml.nvmlDeviceGetMemoryInfo(gpu_handle)
    return info.used / (1024**3)

def dir_size_gb(path: Path):
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024**3)

def iter_text_batches(parquet_files, doc_limit=None, batch_size=2000):
    yielded = 0

    for fp in parquet_files:
        pf = pq.ParquetFile(fp)

        for rb in pf.iter_batches(
            batch_size=batch_size,
            columns=["doc_id", "text_light_clean", "n_words", "n_chars"]
        ):
            table = pa.Table.from_batches([rb])
            df = table.to_pandas()

            if doc_limit is not None:
                remain = doc_limit - yielded
                if remain <= 0:
                    return
                df = df.iloc[:remain].copy()

            if len(df) == 0:
                continue

            yielded += len(df)
            yield df

            if doc_limit is not None and yielded >= doc_limit:
                return

NVMLError_LibraryNotFound: NVML Shared Library Not Found

In [ ]:
# %%
NEIGHBOR_SCHEMA = pa.schema([
    ("query_doc_id", pa.int64()),
    ("neighbor_doc_id", pa.int64()),
    ("score", pa.float32()),
    ("rank", pa.int32()),
])

def write_neighbor_chunk(rows, part_idx: int, out_dir: Path):
    table = pa.Table.from_pylist(rows, schema=NEIGHBOR_SCHEMA)
    out_path = out_dir / f"part-{part_idx:05d}.parquet"
    pq.write_table(table, out_path, compression="zstd")
    return out_path

In [ ]:
# %%
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 256   # engineering choice
model.eval()

In [ ]:
# %%
parquet_files = sorted(TEXT_OUT_DIR.glob("part-*.parquet"))
print("num parquet files:", len(parquet_files))

In [ ]:
# %%
encode_t0 = time.perf_counter()

encode_time = 0.0
num_docs_encoded = 0
peak_ram_gb = ram_used_gb()
peak_gpu_gb = gpu_used_gb()

chunk_idx = 0

for df in tqdm(
    iter_text_batches(parquet_files, doc_limit=RUN_DOC_LIMIT, batch_size=PARQUET_READ_BATCH),
    total=math.ceil(RUN_DOC_LIMIT / PARQUET_READ_BATCH)
):
    texts = [make_embed_view(t) for t in df["text_light_clean"].tolist()]

    t1 = time.perf_counter()
    emb = model.encode(
        texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=NORMALIZE_EMBEDDINGS,
    )
    encode_time += time.perf_counter() - t1

    emb = emb.astype("float32")
    doc_ids_chunk = df["doc_id"].to_numpy(dtype=np.int64)

    np.save(EMB_DIR / f"emb_{chunk_idx:05d}.npy", emb)
    np.save(EMB_DIR / f"doc_ids_{chunk_idx:05d}.npy", doc_ids_chunk)

    num_docs_encoded += len(df)
    peak_ram_gb = max(peak_ram_gb, ram_used_gb())
    peak_gpu_gb = max(peak_gpu_gb, gpu_used_gb())

    del df, texts, emb, doc_ids_chunk
    gc.collect()

    chunk_idx += 1

encode_wall = time.perf_counter() - encode_t0

print("num_docs_encoded:", num_docs_encoded)
print("encode_time_sec :", round(encode_time, 2))
print("encode_wall_sec :", round(encode_wall, 2))
print("embedding_size_gb:", round(dir_size_gb(EMB_DIR), 3))
print("peak_ram_gb:", round(peak_ram_gb, 3))
print("peak_gpu_gb:", round(peak_gpu_gb, 3))

In [ ]:
# %%
emb_files = sorted(EMB_DIR.glob("emb_*.npy"))
id_files = sorted(EMB_DIR.glob("doc_ids_*.npy"))

X = np.concatenate([np.load(f) for f in emb_files], axis=0).astype("float32")
doc_ids = np.concatenate([np.load(f) for f in id_files], axis=0).astype("int64")

print("X shape    :", X.shape)
print("doc_ids    :", doc_ids.shape)
print("RAM now GB :", round(ram_used_gb(), 3))

In [ ]:
# %%
@torch.no_grad()
def topk_search_blockwise_gpu(
    X_np,
    doc_ids_np,
    topk=10,
    query_block_size=1024,
    ref_block_size=8192,
    out_dir=None,
):
    n, d = X_np.shape
    part_idx = 0

    search_t0 = time.perf_counter()
    local_peak_ram = ram_used_gb()
    local_peak_gpu = gpu_used_gb()

    for q0 in tqdm(range(0, n, query_block_size), total=math.ceil(n / query_block_size)):
        q1 = min(q0 + query_block_size, n)
        q = torch.from_numpy(X_np[q0:q1]).to(device, non_blocking=True)

        cur_scores = torch.full((q1 - q0, topk), -1e9, device=device)
        cur_indices = torch.full((q1 - q0, topk), -1, dtype=torch.long, device=device)

        for r0 in range(0, n, ref_block_size):
            r1 = min(r0 + ref_block_size, n)
            r = torch.from_numpy(X_np[r0:r1]).to(device, non_blocking=True)

            sim = q @ r.T   # cosine vì đã normalize

            # mask self-match nếu block overlap
            q_idx = torch.arange(q0, q1, device=device).unsqueeze(1)
            r_idx = torch.arange(r0, r1, device=device).unsqueeze(0)
            sim = sim.masked_fill(q_idx == r_idx, -1e9)

            k_local = min(topk, r1 - r0)
            local_scores, local_pos = torch.topk(sim, k=k_local, dim=1)
            local_indices = local_pos + r0

            merged_scores = torch.cat([cur_scores, local_scores], dim=1)
            merged_indices = torch.cat([cur_indices, local_indices], dim=1)

            new_scores, new_pos = torch.topk(merged_scores, k=topk, dim=1)
            new_indices = torch.gather(merged_indices, 1, new_pos)

            cur_scores = new_scores
            cur_indices = new_indices

            local_peak_ram = max(local_peak_ram, ram_used_gb())
            local_peak_gpu = max(local_peak_gpu, gpu_used_gb())

            del r, sim, local_scores, local_pos, local_indices
            del merged_scores, merged_indices, new_scores, new_pos, new_indices
            gc.collect()

        best_scores = cur_scores.cpu().numpy()
        best_indices = cur_indices.cpu().numpy()

        rows = []
        for i in range(q1 - q0):
            q_doc_id = int(doc_ids_np[q0 + i])
            for rank in range(topk):
                idx = int(best_indices[i, rank])
                if idx < 0:
                    continue
                rows.append({
                    "query_doc_id": q_doc_id,
                    "neighbor_doc_id": int(doc_ids_np[idx]),
                    "score": float(best_scores[i, rank]),
                    "rank": rank + 1,
                })

        if out_dir is not None:
            write_neighbor_chunk(rows, part_idx, out_dir)
            part_idx += 1

        del q, cur_scores, cur_indices, best_scores, best_indices, rows
        gc.collect()

    search_time = time.perf_counter() - search_t0
    return search_time, local_peak_ram, local_peak_gpu

In [ ]:
# %%
search_time, peak_ram_after_search, peak_gpu_after_search = topk_search_blockwise_gpu(
    X_np=X,
    doc_ids_np=doc_ids,
    topk=TOPK,
    query_block_size=QUERY_BLOCK_SIZE,
    ref_block_size=REF_BLOCK_SIZE,
    out_dir=NN_DIR,
)

peak_ram_gb = max(peak_ram_gb, peak_ram_after_search)
peak_gpu_gb = max(peak_gpu_gb, peak_gpu_after_search)

print("search_time_sec:", round(search_time, 2))
print("peak_ram_gb    :", round(peak_ram_gb, 3))
print("peak_gpu_gb    :", round(peak_gpu_gb, 3))
print("neighbors_size_gb:", round(dir_size_gb(NN_DIR), 3))

In [ ]:
# %%
neighbor_files = sorted(NN_DIR.glob("part-*.parquet"))
print("num neighbor parquet files:", len(neighbor_files))

sample_neighbors = pq.read_table(neighbor_files[0]).to_pandas()
display(sample_neighbors.head(20))

In [ ]:
# %%
total_time = encode_wall + search_time

metrics = {
    "run_name": RUN_NAME,
    "pipeline_type": "MiniLM + brute_force_cosine_gpu",
    "model_name": MODEL_NAME,
    "doc_limit": int(RUN_DOC_LIMIT),
    "embedding_dim": int(X.shape[1]),
    "topk": int(TOPK),

    "encode_time_sec": float(encode_time),
    "encode_wall_sec": float(encode_wall),
    "index_time_sec": 0.0,
    "search_time_sec": float(search_time),
    "total_time_sec": float(total_time),

    "observed_peak_ram_gb": float(peak_ram_gb),
    "observed_peak_gpu_gb": float(peak_gpu_gb),

    "text_parquet_size_gb": float(round(dir_size_gb(TEXT_OUT_DIR), 6)),
    "embedding_size_gb": float(round(dir_size_gb(EMB_DIR), 6)),
    "neighbors_size_gb": float(round(dir_size_gb(NN_DIR), 6)),
    "run_output_size_gb": float(round(dir_size_gb(RUN_DIR), 6)),
}

metrics_path = RUN_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

In [ ]:
# %%
import os
import json
import math
import shutil
import time
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

In [ ]:
# %%
def dir_size_bytes(path: Path) -> int:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())

def count_rows_in_parquet_dir(parquet_dir: Path) -> int:
    total = 0
    for fp in sorted(parquet_dir.glob("part-*.parquet")):
        pf = pq.ParquetFile(fp)
        total += pf.metadata.num_rows
    return total

def reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

In [ ]:
# %%
parquet_files = sorted(TEXT_OUT_DIR.glob("part-*.parquet"))

print("num_files     :", len(parquet_files))
print("total_rows    :", count_rows_in_parquet_dir(TEXT_OUT_DIR))
print("total_size_GB :", round(dir_size_bytes(TEXT_OUT_DIR) / (1024**3), 3))

sample = pq.read_table(
    parquet_files[0],
    columns=["doc_id", "source", "n_words", "n_chars", "exact_hash"]
).to_pandas()

display(sample.head(10))

In [ ]:
# %%
SCALE_POINTS = [200_000, 400_000, 600_000, 800_000, 1_000_000]
ROWS_PER_SUBSET_FILE = 10_000

SUBSET_ROOT = Path("/kaggle/working/pes2o_scale_text_subsets")
reset_dir(SUBSET_ROOT)

In [ ]:
TEXT_COLUMNS = [
    "doc_id",
    "source",
    "text_light_clean",
    "n_words",
    "n_chars",
    "exact_hash",
]

def iter_text_tables_limited(parquet_dir: Path, doc_limit=None, batch_size=10_000):
    yielded = 0

    for fp in sorted(parquet_dir.glob("part-*.parquet")):
        pf = pq.ParquetFile(fp)

        for rb in pf.iter_batches(batch_size=batch_size, columns=TEXT_COLUMNS):
            table = pa.Table.from_batches([rb])

            if doc_limit is not None:
                remain = doc_limit - yielded
                if remain <= 0:
                    return
                if table.num_rows > remain:
                    table = table.slice(0, remain)

            if table.num_rows == 0:
                continue

            yielded += table.num_rows
            yield table

            if doc_limit is not None and yielded >= doc_limit:
                return

In [ ]:
# %%
TEXT_COLUMNS = [
    "doc_id",
    "source",
    "text_light_clean",
    "n_words",
    "n_chars",
    "exact_hash",
]

def iter_text_tables(parquet_dir: Path, batch_size=10_000):
    for fp in sorted(parquet_dir.glob("part-*.parquet")):
        pf = pq.ParquetFile(fp)
        for rb in pf.iter_batches(batch_size=batch_size, columns=TEXT_COLUMNS):
            yield pa.Table.from_batches([rb])

In [ ]:
# %%
def create_text_subset_from_master(
    master_dir: Path,
    out_dir: Path,
    target_rows: int,
    rows_per_file: int = 10_000,
):
    reset_dir(out_dir)

    written = 0
    file_idx = 0
    buffer_tables = []
    buffer_rows = 0

    for table in iter_text_tables(master_dir, batch_size=rows_per_file):
        remain = target_rows - written
        if remain <= 0:
            break

        if table.num_rows > remain:
            table = table.slice(0, remain)

        buffer_tables.append(table)
        buffer_rows += table.num_rows
        written += table.num_rows

        while buffer_rows >= rows_per_file:
            merged = pa.concat_tables(buffer_tables)
            to_write = merged.slice(0, rows_per_file)

            out_path = out_dir / f"part-{file_idx:05d}.parquet"
            pq.write_table(to_write, out_path, compression="zstd")

            leftover = merged.slice(rows_per_file)
            buffer_tables = [leftover] if leftover.num_rows > 0 else []
            buffer_rows = leftover.num_rows
            file_idx += 1

        if written >= target_rows:
            break

    if buffer_rows > 0:
        merged = pa.concat_tables(buffer_tables)
        out_path = out_dir / f"part-{file_idx:05d}.parquet"
        pq.write_table(merged, out_path, compression="zstd")

    return {
        "target_rows": target_rows,
        "actual_rows": count_rows_in_parquet_dir(out_dir),
        "size_bytes": dir_size_bytes(out_dir),
        "num_files": len(list(out_dir.glob("part-*.parquet"))),
    }

In [ ]:
# %%
subset_summaries = []

for n_docs in SCALE_POINTS:
    subset_dir = SUBSET_ROOT / f"text_{n_docs//1000:03d}k"
    summary = create_text_subset_from_master(
        master_dir=TEXT_OUT_DIR,
        out_dir=subset_dir,
        target_rows=n_docs,
        rows_per_file=ROWS_PER_SUBSET_FILE,
    )
    subset_summaries.append({
        "subset_name": subset_dir.name,
        "n_docs": n_docs,
        "actual_rows": summary["actual_rows"],
        "num_files": summary["num_files"],
        "size_gb": round(summary["size_bytes"] / (1024**3), 4),
    })

display(pd.DataFrame(subset_summaries))

In [ ]:
# %%
for n_docs in SCALE_POINTS:
    subset_dir = SUBSET_ROOT / f"text_{n_docs//1000:03d}k"
    rows = count_rows_in_parquet_dir(subset_dir)
    size_gb = dir_size_bytes(subset_dir) / (1024**3)
    print(subset_dir.name, "rows =", rows, "size_gb =", round(size_gb, 4))

In [ ]:
# %%
check_dir = SUBSET_ROOT / "text_050k"
check_files = sorted(check_dir.glob("part-*.parquet"))

sample_df = pq.read_table(
    check_files[0],
    columns=["doc_id", "n_words", "n_chars", "text_light_clean"]
).to_pandas()

display(sample_df.head(3))

In [ ]:
# %%
BENCH_ROOT = Path("/kaggle/working/pes2o_scale_benchmark_runs")
reset_dir(BENCH_ROOT)

In [ ]:
# %%
def benchmark_one_subset(
    subset_dir: Path,
    algo_name: str,
    runner_fn,
    bench_root: Path,
    extra_config: dict | None = None,
):
    run_dir = bench_root / f"{algo_name}__{subset_dir.name}"
    reset_dir(run_dir)

    t0 = time.perf_counter()
    runner_result = runner_fn(subset_dir=subset_dir, out_dir=run_dir)
    wall_clock_sec = time.perf_counter() - t0

    metrics = {
        "algo_name": algo_name,
        "subset_name": subset_dir.name,
        "n_docs": count_rows_in_parquet_dir(subset_dir),
        "wall_clock_sec": wall_clock_sec,
        "disk_usage_bytes": dir_size_bytes(run_dir),
    }

    if extra_config is not None:
        metrics["config"] = extra_config

    if isinstance(runner_result, dict):
        metrics.update(runner_result)

    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    return metrics

In [ ]:
# %%
def smoke_test_runner(subset_dir: Path, out_dir: Path):
    total_docs = 0
    total_words = 0

    for table in iter_text_tables(subset_dir, batch_size=5000):
        df = table.to_pandas()
        total_docs += len(df)
        total_words += int(df["n_words"].sum())

    stats = {
        "total_docs_seen": total_docs,
        "total_words_seen": total_words,
        "avg_words": total_words / max(total_docs, 1),
    }

    with open(out_dir / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    return stats

In [ ]:
# %%
smoke_metrics = []

for n_docs in SCALE_POINTS:
    subset_dir = SUBSET_ROOT / f"text_{n_docs//1000:03d}k"
    m = benchmark_one_subset(
        subset_dir=subset_dir,
        algo_name="smoke_test",
        runner_fn=smoke_test_runner,
        bench_root=BENCH_ROOT,
    )
    smoke_metrics.append(m)

display(pd.DataFrame(smoke_metrics))

In [ ]:
# %%
def fit_linear_trend(x, y):
    coef = np.polyfit(x, y, deg=1)   # y = a*x + b
    a, b = coef[0], coef[1]
    return a, b

smoke_df = pd.DataFrame(smoke_metrics).sort_values("n_docs")

a_time, b_time = fit_linear_trend(
    smoke_df["n_docs"].to_numpy(),
    smoke_df["wall_clock_sec"].to_numpy()
)

a_disk, b_disk = fit_linear_trend(
    smoke_df["n_docs"].to_numpy(),
    smoke_df["disk_usage_bytes"].to_numpy()
)

print("time fit: y =", a_time, "* n_docs +", b_time)
print("disk fit: y =", a_disk, "* n_docs +", b_disk)

In [ ]:
# %%
def extrapolate(metric_slope, metric_bias, n_docs):
    return metric_slope * n_docs + metric_bias

for target in [500_000, 1_000_000]:
    pred_time = extrapolate(a_time, b_time, target)
    pred_disk = extrapolate(a_disk, b_disk, target)

    print(
        f"target={target:,} docs | "
        f"pred_time_sec={pred_time:.2f} | "
        f"pred_disk_GB={pred_disk/(1024**3):.4f}"
    )

In [ ]:
# %%
!pip -q install datasketch psutil

In [ ]:
# %%
import os
import re
import gc
import json
import math
import time
import pickle
import psutil
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from datasketch import MinHash, MinHashLSH
from tqdm.auto import tqdm

In [ ]:
# %%
MINHASH_NUM_PERM = 64
MINHASH_THRESHOLD = 0.80

MINHASH_BATCH_ROWS = 2000
SAVE_SAMPLE_HITS = 1000

TOKEN_RE = re.compile(r"\w+")
process = psutil.Process(os.getpid())

print("MINHASH_NUM_PERM =", MINHASH_NUM_PERM)
print("MINHASH_THRESHOLD =", MINHASH_THRESHOLD)
print("NGRAM_N =", NGRAM_N)
print("MAX_UNIQUE_SHINGLES =", MAX_UNIQUE_SHINGLES)

In [ ]:
# %%
def ram_used_gb():
    return process.memory_info().rss / (1024**3)

def fast_hash_32(x: bytes) -> int:
    return xxhash.xxh32_intdigest(x)

def tokenize_for_minhash(text: str):
    return TOKEN_RE.findall(text.lower())

def iter_hashed_shingles(tokens, n=NGRAM_N, max_unique=MAX_UNIQUE_SHINGLES):
    seen = set()

    if not tokens:
        return

    if len(tokens) < n:
        sh = " ".join(tokens).encode("utf-8", errors="ignore")
        h = xxhash.xxh64_digest(sh)
        yield h
        return

    for i in range(len(tokens) - n + 1):
        sh = " ".join(tokens[i:i+n]).encode("utf-8", errors="ignore")
        h = xxhash.xxh64_digest(sh)

        if h in seen:
            continue

        seen.add(h)
        yield h

        if len(seen) >= max_unique:
            break

def build_minhash_from_text(
    text: str,
    num_perm=MINHASH_NUM_PERM,
    ngram_n=NGRAM_N,
    max_unique=MAX_UNIQUE_SHINGLES,
):
    tokens = tokenize_for_minhash(text)

    mh = MinHash(num_perm=num_perm, hashfunc=fast_hash_32)

    n_shingles = 0
    for sh in iter_hashed_shingles(tokens, n=ngram_n, max_unique=max_unique):
        mh.update(sh)
        n_shingles += 1

    if n_shingles == 0:
        mh.update(b"__EMPTY__")
        n_shingles = 1

    return mh, len(tokens), n_shingles

In [ ]:
# %%
SAMPLE_HIT_SCHEMA = pa.schema([
    ("query_doc_id", pa.int64()),
    ("matched_doc_id", pa.int64()),
])

def save_sample_hits(sample_hits, out_path: Path):
    if not sample_hits:
        return
    table = pa.Table.from_pylist(sample_hits, schema=SAMPLE_HIT_SCHEMA)
    pq.write_table(table, out_path, compression="zstd")

In [ ]:
# %%
SAMPLE_HIT_SCHEMA = pa.schema([
    ("query_doc_id", pa.int64()),
    ("matched_doc_id", pa.int64()),
])

def save_sample_hits(sample_hits, out_path: Path):
    if not sample_hits:
        return
    table = pa.Table.from_pylist(sample_hits, schema=SAMPLE_HIT_SCHEMA)
    pq.write_table(table, out_path, compression="zstd")

In [ ]:
# %%
def minhash_lsh_runner(subset_dir: Path, out_dir: Path):
    sig_dir = out_dir / "signatures"
    sig_dir.mkdir(parents=True, exist_ok=True)

    lsh = MinHashLSH(
        threshold=MINHASH_THRESHOLD,
        num_perm=MINHASH_NUM_PERM,
    )

    n_docs = 0
    total_tokens = 0
    total_shingles = 0

    docs_with_hits = 0
    total_hit_count = 0

    build_sig_sec = 0.0
    query_sec = 0.0
    insert_sec = 0.0

    peak_ram_gb = ram_used_gb()
    sample_hits = []

    part_idx = 0

    for table in tqdm(iter_text_tables(subset_dir, batch_size=MINHASH_BATCH_ROWS)):
        df = table.select(["doc_id", "text_light_clean"]).to_pandas()

        batch_doc_ids = []
        batch_hashvalues = []

        for row in df.itertuples(index=False):
            doc_id = int(row.doc_id)
            text = row.text_light_clean

            t0 = time.perf_counter()
            mh, n_tokens, n_shingles = build_minhash_from_text(text)
            build_sig_sec += time.perf_counter() - t0

            t0 = time.perf_counter()
            hits = lsh.query(mh)
            query_sec += time.perf_counter() - t0

            if hits:
                docs_with_hits += 1
                total_hit_count += len(hits)

                if len(sample_hits) < SAVE_SAMPLE_HITS:
                    for h in hits:
                        sample_hits.append({
                            "query_doc_id": doc_id,
                            "matched_doc_id": int(h),
                        })
                        if len(sample_hits) >= SAVE_SAMPLE_HITS:
                            break

            t0 = time.perf_counter()
            lsh.insert(str(doc_id), mh)
            insert_sec += time.perf_counter() - t0

            batch_doc_ids.append(doc_id)
            batch_hashvalues.append(mh.hashvalues.astype(np.uint64))

            n_docs += 1
            total_tokens += n_tokens
            total_shingles += n_shingles
            peak_ram_gb = max(peak_ram_gb, ram_used_gb())

        np.save(sig_dir / f"doc_ids_{part_idx:05d}.npy",
                np.asarray(batch_doc_ids, dtype=np.int64))
        np.save(sig_dir / f"minhash_{part_idx:05d}.npy",
                np.vstack(batch_hashvalues).astype(np.uint64))

        part_idx += 1

        del df, batch_doc_ids, batch_hashvalues
        gc.collect()

    index_pickle_ok = True
    index_pickle_error = None

    try:
        with open(out_dir / "lsh_index.pkl", "wb") as f:
            pickle.dump(lsh, f, protocol=pickle.HIGHEST_PROTOCOL)
    except Exception as e:
        index_pickle_ok = False
        index_pickle_error = repr(e)
        with open(out_dir / "lsh_index_pickle_error.txt", "w") as f:
            f.write(index_pickle_error)

    save_sample_hits(sample_hits, out_dir / "sample_hits.parquet")

    stats = {
        "runner_type": "MinHashLSH",
        "minhash_num_perm": int(MINHASH_NUM_PERM),
        "minhash_threshold": float(MINHASH_THRESHOLD),
        "ngram_n": int(NGRAM_N),
        "max_unique_shingles": int(MAX_UNIQUE_SHINGLES),

        "total_docs_seen": int(n_docs),
        "total_tokens": int(total_tokens),
        "total_shingles_kept": int(total_shingles),

        "docs_with_hits": int(docs_with_hits),
        "total_hit_count": int(total_hit_count),

        "build_signature_sec": float(build_sig_sec),
        "query_sec": float(query_sec),
        "insert_sec": float(insert_sec),

        "peak_ram_gb": float(peak_ram_gb),

        "index_pickle_ok": bool(index_pickle_ok),
    }

    if index_pickle_error is not None:
        stats["index_pickle_error"] = index_pickle_error

    with open(out_dir / "runner_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    return stats

In [ ]:
# %%
minhash_metrics = []

for n_docs in SCALE_POINTS:
    subset_dir = SUBSET_ROOT / f"text_{n_docs//1000:03d}k"

    m = benchmark_one_subset(
        subset_dir=subset_dir,
        algo_name="minhash_lsh",
        runner_fn=minhash_lsh_runner,
        bench_root=BENCH_ROOT,
        extra_config={
            "num_perm": MINHASH_NUM_PERM,
            "threshold": MINHASH_THRESHOLD,
            "ngram_n": NGRAM_N,
            "max_unique_shingles": MAX_UNIQUE_SHINGLES,
        },
    )
    minhash_metrics.append(m)

minhash_df = pd.DataFrame(minhash_metrics).sort_values("n_docs")
minhash_df["disk_usage_gb"] = minhash_df["disk_usage_bytes"] / (1024**3)

display(minhash_df[[
    "subset_name",
    "n_docs",
    "wall_clock_sec",
    "disk_usage_bytes",
    "disk_usage_gb",
    "build_signature_sec",
    "query_sec",
    "insert_sec",
    "peak_ram_gb",
    "docs_with_hits",
    "total_hit_count",
]])

In [ ]:
# %%
minhash_df["sig_pct"] = minhash_df["build_signature_sec"] / minhash_df["wall_clock_sec"]
minhash_df["query_pct"] = minhash_df["query_sec"] / minhash_df["wall_clock_sec"]
minhash_df["insert_pct"] = minhash_df["insert_sec"] / minhash_df["wall_clock_sec"]

display(minhash_df[[
    "subset_name",
    "n_docs",
    "wall_clock_sec",
    "sig_pct",
    "query_pct",
    "insert_pct",
]])

In [ ]:
# %%
def fit_linear_trend(x, y):
    coef = np.polyfit(x, y, deg=1)
    a, b = coef[0], coef[1]
    return a, b

a_time, b_time = fit_linear_trend(
    minhash_df["n_docs"].to_numpy(),
    minhash_df["wall_clock_sec"].to_numpy()
)

a_disk, b_disk = fit_linear_trend(
    minhash_df["n_docs"].to_numpy(),
    minhash_df["disk_usage_bytes"].to_numpy()
)

print("time fit : y =", a_time, "* n_docs +", b_time)
print("disk fit : y =", a_disk, "* n_docs +", b_disk)

In [ ]:
# %%
def extrapolate(metric_slope, metric_bias, n_docs):
    return metric_slope * n_docs + metric_bias

for target in [500_000, 1_000_000]:
    pred_time = extrapolate(a_time, b_time, target)
    pred_disk = extrapolate(a_disk, b_disk, target)

    print(
        f"target={target:,} docs | "
        f"pred_time_sec={pred_time:.2f} | "
        f"pred_disk_GB={pred_disk/(1024**3):.4f}"
    )

In [ ]:
# %%
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(minhash_df["n_docs"], minhash_df["wall_clock_sec"], marker="o")
plt.xlabel("Number of documents")
plt.ylabel("Wall-clock time (sec)")
plt.title("MinHashLSH runtime scaling on peS2o subsets")
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(minhash_df["n_docs"], minhash_df["disk_usage_gb"], marker="o")
plt.xlabel("Number of documents")
plt.ylabel("Disk usage (GB)")
plt.title("MinHashLSH disk scaling on peS2o subsets")
plt.grid(True)
plt.show()

In [ ]:
# %%
final_scale_table = minhash_df[[
    "subset_name",
    "n_docs",
    "wall_clock_sec",
    "disk_usage_bytes",
    "disk_usage_gb",
    "build_signature_sec",
    "query_sec",
    "insert_sec",
    "peak_ram_gb",
    "docs_with_hits",
    "total_hit_count",
]].copy()

final_scale_table.to_csv(BENCH_ROOT / "minhash_lsh_scale_results.csv", index=False)
display(final_scale_table)